In [5]:
import numpy as np
import pandas as pd
import pickle

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error

In [6]:
with open("../models/preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

with open("../models/feature_order.pkl", "rb") as f:
    feature_order = pickle.load(f)

num_features = feature_order["numeric"]
cat_features = feature_order["categorical"]

FileNotFoundError: [Errno 2] No such file or directory: '../models/preprocessor.pkl'

In [ ]:
df = pd.read_csv("../data/raw/housing.csv")

df["rooms_per_household"] = (df["total_rooms"] / df["households"]).clip(0, 20)
df["bedrooms_per_room"] = (df["total_bedrooms"] / df["total_rooms"]).clip(0, 1)

for col in ["total_rooms", "total_bedrooms", "population", "households", "median_income"]:
    df[f"{col}_log1p"] = np.log1p(df[col])

df = df.drop(columns=["total_rooms", "total_bedrooms", "population", "households", "median_income"])

In [ ]:
target = "median_house_value"

X = df.drop(columns=[target])
y_log = np.log1p(df[target]).astype(float)

y_mean = y_log.mean()
y_std  = y_log.std()
y = (y_log - y_mean) / y_std

print("y mean (log):", y_mean, "y std (log):", y_std)

In [ ]:
X_proc = preprocessor.transform(X)

print("X_proc shape:", X_proc.shape)
print("Max abs value (sanity):", np.max(np.abs(X_proc)))

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_proc, y, test_size=0.3, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42
)

print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
ones_X = np.ones((X_train.shape[0], 1))
ones_X_val = np.ones((X_val.shape[0], 1))
ones_X_test = np.ones((X_test.shape[0], 1))

X_train_bias = np.c_[ones_X, X_train]
X_val_bias = np.c_[ones_X_val, X_val]
X_test_bias = np.c_[ones_X_test, X_test]

In [ ]:
def predict(X, theta):
    return X @ theta

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred) ** 2))

def compute_cost_ridge(X, y, theta, lmbd):
    m = len(y)
    err = predict(X, theta) - y
    cost = (1 / (2 * m)) * np.sum(err ** 2)
    reg = (lmbd / (2 * m)) * np.sum(theta[1:] ** 2)
    return cost + reg

def compute_grad_ridge(X, y, theta, lmbd):
    m = len(y)
    err = predict(X, theta) - y
    grad = (1 / m) * (X.T @ err)
    grad[1:] += (lmbd / m) * theta[1:]
    return grad

def train_ridge_gd(X, y, lmbd, alpha, iters):
    theta = np.zeros(X.shape[1])
    for _ in range(iters):
        theta -= alpha * compute_grad_ridge(X, y, theta, lmbd)
    return theta

In [ ]:
theta_linear = train_ridge_gd(X_train_bias, y_train, 0.0, 0.0005, 3000)

In [ ]:
lambdas = [0.01, 0.1, 1, 10, 50, 60]

results = []

for lmbd in lambdas:
    theta = train_ridge_gd(X_train_bias, y_train, lmbd, 0.0005, 3000)
    
    train_rmse = rmse(y_train, predict(X_train_bias, theta))
    val_rmse   = rmse(y_val, predict(X_val_bias, theta))
    
    results.append((lmbd, train_rmse, val_rmse))

In [ ]:
results_df = pd.DataFrame(
    results, columns=["lambda", "train_rmse", "val_rmse"]
)

baseline_rmse = rmse(y_val, predict(X_val_bias, theta_linear))
results_df.loc[-1] = [0.0, rmse(y_train, predict(X_train_bias, theta_linear)), baseline_rmse]
results_df = results_df.sort_values("lambda")

results_df

In [ ]:
plt.figure()
plt.semilogx(results_df["lambda"], results_df["val_rmse"], marker="o")
plt.xlabel("Lambda (log scale)")
plt.ylabel("Validation RMSE (scaled)")
plt.title("Validation Curve for Ridge Regression")
plt.show()

In [ ]:
best_lambda = results_df.iloc[results_df["val_rmse"].idxmin()]["lambda"]
print("Best lambda:", best_lambda)

theta_final = train_ridge_gd(X_train_bias, y_train, best_lambda, 0.0005, 3000)

test_rmse_scaled = rmse(y_test, predict(X_test_bias, theta_final))
print("Test RMSE (scaled):", test_rmse_scaled)

In [ ]:
def inverse_target(y_scaled):
    return np.expm1(y_scaled * y_std + y_mean)

y_test_real = inverse_target(y_test)
y_test_pred_real = inverse_target(predict(X_test_bias, theta_final))

test_rmse_real = np.sqrt(mean_squared_error(y_test_real, y_test_pred_real))
print("Test RMSE (real):", test_rmse_real)

In [18]:
np.save("../models/theta_final.npy", theta_final)

with open("../models/target_scaler.pkl", "wb") as f:
    pickle.dump({"mean": y_mean, "std": y_std}, f)

with open("../models/best_lambda.txt", "w") as f:
    f.write(str(best_lambda))